### MCP(Model Context Protocol)
- 남이 만들어둔 서버에 붙어서 도구를 받아올 수 있다.
- 공통 규격을 말함
- 받아오는 쪽!
- 비동기 : 요청 보내놓고, 다른 일을 할 수 있는 방식 -> async

In [2]:
import asyncio
import nest_asyncio

In [3]:
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent

### 1. 서버 - 내 컴퓨터에서 띄우기
- 우리 컴퓨터에서 별도로 띄우고, 외부에서 도구 가져오자!

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

In [ ]:
# 현대 시간 반환하는 time 도구 가져오기

local_client = MultiServerMCPClient({
    "time" : {
        "transport" : "stdio",
        "command" : "uvx",
        "args" : ['mcp-server-time']
    }
})

async def get_local_tools():    # def 앞에 async 붙이면 return 전 await 붙여야 한다!
    return await local_client.get_tools()

local_tools = await get_local_tools()    # 비동기일 때 함수 시행하는 방법! (이건 주피터 노트북이라 그럼.?)
[tool.name for tool in local_tools]

['get_current_time', 'convert_time']

In [10]:
# tools = local_tools

# 1. 모델 설정
model = init_chat_model("openai:gpt-5.6-luna", reasoning_effort="none")

# 2. agent 설정
agent = create_agent(
    model=model,
    tools=local_tools,
    system_prompt="너는 비서다. 도구로 확인한 내용을 답해라."
)

In [20]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

# agent 호출용 함수 - 비동기!
async def ask(question):
    response = await agent.ainvoke({"messages" : [ HumanMessage(question) ]}) #awake 넣고, invoke가 아니라 비동기라 ainvoke인 것 주목!!
    return response

In [17]:
result = await ask("안녕하세요")
result['messages']

/tmp/ipykernel_130279/1757339481.py:1: RuntimeWarning: coroutine 'ask' was never awaited
  result = await ask("안녕하세요")


[HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='278a6b53-4cfd-4ea0-b4d3-8c947ad9e383'),
 AIMessage(content='안녕하세요! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 311, 'total_tokens': 325, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ER9qrIn8zvhe24PfUdwF8p91Bsxkl', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ccbe-8680-70b1-a342-61490f6be95a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 311, 'output_tokens': 14, 'total_tokens': 325, 'input_token_details': {'audio

In [21]:
result = await ask("현재 시간은?")
result['messages']

/tmp/ipykernel_130279/1640016642.py:1: RuntimeWarning: coroutine 'Pregel.ainvoke' was never awaited
  result = await ask("현재 시간은?")


[HumanMessage(content='현재 시간은?', additional_kwargs={}, response_metadata={}, id='2743f766-5a33-47d2-bda5-da508f4e2057'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 313, 'total_tokens': 334, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ER9syWR2Gri1GtGtviGxo4gMVQArx', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a0ccc0-8538-7f40-8d5b-a92657530657-0', tool_calls=[{'name': 'get_current_time', 'args': {'timezone': 'Asia/Seoul'}, 'id': 'call_Vhg5VO39cDuii5q2a9iNAZFV', 'type': 'tool_call'}], invalid_tool_calls

In [ ]:
# 도구 추가하기 - 웹서치 - 페이지 내용 가져오기 - fetch

# fetch : 웹서치해서 md로 가져오는 코드?

local_client = MultiServerMCPClient({
    "time" : {
        "transport" : "stdio",
        "command" : "uvx",
        "args" : ['mcp-server-time']
    },

    "fetch" : {
        "transport" : "stdio",
        "command" : "uvx",
        "args" : ['mcp-server-fetch']
    }
})

async def get_local_tools():    # def 앞에 async 붙이면 return 전 await 붙여야 한다!
    return await local_client.get_tools()

local_tools = await get_local_tools()    # 비동기일 때 함수 시행하는 방법! (이건 주피터 노트북이라 그럼.?)
local_tools

[StructuredTool(name='get_current_time', description='Get current time in a specific timezone', args_schema={'type': 'object', 'properties': {'timezone': {'type': 'string', 'description': "IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'Asia/Seoul' as local timezone if no timezone provided by the user."}}, 'required': ['timezone']}, metadata={'title': None, 'readOnlyHint': True, 'destructiveHint': False, 'idempotentHint': True, 'openWorldHint': False}, handle_tool_error=<function _handle_mcp_tool_error at 0x79b8dd2725c0>, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x79b8dba2c900>),
 StructuredTool(name='convert_time', description='Convert time between timezones', args_schema={'type': 'object', 'properties': {'source_timezone': {'type': 'string', 'description': "Source IANA timezone name (e.g., 'America/New_York', 'Europe/London'). Use 'Asia/Seoul' as local timezone if no source timezone provi

In [26]:
# 도구 추가해 agent 재설정
agent = create_agent(
    model=model,
    tools=local_tools,
    system_prompt="너는 비서다. 도구로 확인한 내용을 답해라."
)

In [27]:
# 실제 웹페이지 읽어 md로 만들기 - 나무위키 'MCP' 페이지
question = r"https://namu.wiki/w/%EB%B6%81%ED%95%AD%EC%B9%9C%EC%88%98%EA%B3%B5%EC%9B%90%20%EC%83%81%EC%96%B4%20%EC%B6%9C%EB%AA%B0%20%EC%82%AC%EA%B1%B4?from=%EB%B6%80%EC%82%B0%20%EC%83%81%EC%96%B4%20%EC%B6%9C%EB%AA%B0%20%EC%82%AC%EA%B1%B4 이 페이지 읽고 내용 정리해 줘. 웹페이지 안읽어지면 안읽어진다고 말해."

result = await ask(question)
result['messages']

[HumanMessage(content='https://namu.wiki/w/%EB%B6%81%ED%95%AD%EC%B9%9C%EC%88%98%EA%B3%B5%EC%9B%90%20%EC%83%81%EC%96%B4%20%EC%B6%9C%EB%AA%B0%20%EC%82%AC%EA%B1%B4?from=%EB%B6%80%EC%82%B0%20%EC%83%81%EC%96%B4%20%EC%B6%9C%EB%AA%B0%20%EC%82%AC%EA%B1%B4 이 페이지 읽고 내용 정리해 줘. 웹페이지 안읽어지면 안읽어진다고 말해.', additional_kwargs={}, response_metadata={}, id='fc29cb75-102d-4a66-8c57-de2cee38e6fd'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 166, 'prompt_tokens': 665, 'total_tokens': 831, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'chatcmpl-ERA3oJzBzj4V27rDMKq3x6wTE28BV', 'service_tier': 'default', '

In [28]:
print(result['messages'][-1].content)

페이지는 읽을 수 있었습니다. 다만 **나무위키 문서의 내용은 검증되지 않았거나 편향·오류가 있을 수 있다**는 점을 전제로 요약합니다.

## 북항친수공원 상어 출몰 사건 요약

### 1. 사건 개요
- **2026년 9월 18일** 부산광역시 동구 **북항친수공원 수로**에서 대형 상어가 발견됨.
- 최초 신고 시각은 **오전 8시 57분경**.
- 상어의 크기는 약 **3~3.5m**로 추정됨.
- 종은 정확히 확인되지 않았으나 문서에서는 **무태상어 또는 흑상어**일 가능성을 언급함.
- 북항친수공원 수로는 민물이 아니라 바닷물과 연결된 해수 수로임.

### 2. 상어의 행동과 대응
- 해경은 처음에 상어를 외해로 유도하려 했지만, 자극할 경우 공격성을 보일 수 있다는 전문가 의견에 따라 무리하게 쫓아내지 않고 자연스럽게 빠져나가기를 기다리기로 함.
- 그러나 상어는 수로를 떠나지 않고 며칠간 머물렀음.
- 사람들 가까이 다가오거나 물 밖으로 뛰어오르는 모습이 목격되면서 온라인에서 화제가 됨.
- 전문가들은 상어가 수로 벽 쪽에 머무는 이유로:
  - 벽면이 있어 사방에서 공격받을 가능성이 낮고,
  - 수로 벽 주변에 먹이가 될 만한 물고기가 많기 때문일 수 있다고 설명함.
- 상어가 장기간 머물 경우 차단망을 설치해 외해로 유도하는 방안도 검토됨.

### 3. 관광객과 주변 상권
- 언론 보도와 부산시의 안전안내문자 이후 상어를 보려는 사람들이 급격히 몰림.
- 주말에는 주차장이 만석이 되고 경찰이 펜스를 설치해 관람객을 통제함.
- 인근의 거의 유일한 카페인 **이디야 부산북항친수공원점**은 손님이 폭증해 직원과 가족까지 동원할 정도로 바빠짐.
- 푸드트럭도 현장 주변으로 몰려들었으며, 북항친수공원은 의도하지 않게 큰 홍보 효과를 얻음.
- 문서에 따르면 사건 발생 후 사흘 동안 약 **3만 명**이 방문했다고 함.

### 4. 새끼 상어 논란
- 9월 20일경 약 1m 크기의 개체가 추가로 발견됐다는 이야기가 퍼지며 ‘상어가 새끼를 낳은 

In [30]:
# 주어진 이름과 tool과 시스템 프롬프트 가지는 agent 만드는 함수!
def make_agent(agent_name, tools, system_prompt):
    agent = create_agent(
        model=model,
        tools=tools,
        system_prompt=system_prompt
    )

    return agent

### MCP 서버가 내 컴퓨터가 아닌 원격에 있는 경우
- 인터넷에 떠있는 서버 주소만 알면 가져와서 에이전트로 구성할 수 있다!
- streamable http 방식
- deepwiki: 깃허브 저장소 문서를 읽고 질문에 대답해주는 공개 서버
- api키가 있는 경우가 많다.

In [ ]:
remote_client = MultiServerMCPClient({
    "deepwiki" : {
        "transport" : "streamable_http",
        "url" : "http://mcp.deepwiki.com/mcp"
    }
})

async def get_remote_tools():
    return await remote_client.get_tools()

remote_tools = await get_remote_tools()
remote_tools

In [ ]:
# remote_client = MultiServerMCPClient({
#     "filesystem": {
#         "transport" : "stdio",
#         "command": "uvx",
#         "args": [
#             "-y",
#             "@modelcontextprotocol/server-filesystem",
#             "/home/playdata",
#         ]
#     }
# })

# async def get_remote_tools():
#     return await remote_client.get_tools()

# remote_tools = await get_remote_tools()
# remote_tools